In [1]:
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Folder containing all CSV files
CSV_FOLDER = "."

CHECKPOINT_PATH = "csv_llm_1M_checkpoint.pth"
WEIGHTS_PATH = "csv_llm_1M_weights.pth"

SEED = 42

# Training settings
batch_size = 16
block_size = 128
training_steps = 2000
eval_interval = 100
eval_iters = 20

learning_rate = 3e-4
weight_decay = 0.01
dropout = 0.1

# Approximately 1M parameter model
embedding_dim = 128
num_heads = 4
num_layers = 5


# ============================================================
# 2. DEVICE
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)


# ============================================================
# 3. RANDOM SEED
# ============================================================

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. FIND ALL CSV FILES
# ============================================================

csv_files = sorted(
    Path(CSV_FOLDER).glob("*.csv")
)

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in folder: {CSV_FOLDER}"
    )


print("\n====================================")
print("CSV FILES FOUND")
print("====================================")

for file in csv_files:
    print(file.name)

print("\nTotal CSV files:", len(csv_files))


# ============================================================
# 5. CHECK COLUMN CONSISTENCY
# ============================================================

reference_file = csv_files[0]

reference_columns = list(
    pd.read_csv(
        reference_file,
        nrows=0
    ).columns
)

reference_column_set = set(
    reference_columns
)


print("\n====================================")
print("COLUMN CHECK")
print("====================================")

print(
    f"Reference file: {reference_file.name}"
)

print(
    f"Reference columns: "
    f"{len(reference_columns)}"
)


all_columns_match = True


for file in csv_files:

    columns = list(
        pd.read_csv(
            file,
            nrows=0
        ).columns
    )

    column_set = set(columns)

    missing_columns = (
        reference_column_set
        - column_set
    )

    extra_columns = (
        column_set
        - reference_column_set
    )

    print(
        f"\n{file.name}: "
        f"{len(columns)} columns"
    )

    if not missing_columns and not extra_columns:

        print("  Columns match")

    else:

        all_columns_match = False

        print("  Column mismatch")

        if missing_columns:
            print(
                "  Missing columns:",
                sorted(missing_columns)
            )

        if extra_columns:
            print(
                "  Additional columns:",
                sorted(extra_columns)
            )


# ============================================================
# 6. STOP IF COLUMN STRUCTURE IS DIFFERENT
# ============================================================

if not all_columns_match:

    raise ValueError(
        "\nSome CSV files have different columns. "
        "Fix the column mismatch before training."
    )


print(
    "\nAll CSV files have compatible columns."
)


# ============================================================
# 7. READ AND MERGE ALL CSV FILES
# ============================================================

dataframes = []

print("\n====================================")
print("READING CSV FILES")
print("====================================")


for file in csv_files:

    temp_df = pd.read_csv(file)

    # Ensure same column order as first CSV
    temp_df = temp_df[
        reference_columns
    ]

    print(
        f"{file.name}: "
        f"{len(temp_df)} rows, "
        f"{len(temp_df.columns)} columns"
    )

    dataframes.append(
        temp_df
    )


df = pd.concat(
    dataframes,
    ignore_index=True
)


print("\n====================================")
print("MERGED DATASET")
print("====================================")

print(
    "Total CSV files:",
    len(csv_files)
)

print(
    "Total rows:",
    len(df)
)

print(
    "Total columns:",
    len(df.columns)
)


print("\nColumn names:")

for col in df.columns:
    print(" -", col)


# ============================================================
# 8. OPTIONAL: SAVE MERGED CSV
# ============================================================

MERGED_CSV_PATH = "merged_training_data.csv"

df.to_csv(
    MERGED_CSV_PATH,
    index=False
)

print(
    "\nMerged CSV saved as:",
    MERGED_CSV_PATH
)


# ============================================================
# 9. CONVERT EACH CSV ROW TO TEXT
# ============================================================

def row_to_text(row):

    parts = []

    for column in df.columns:

        value = row[column]

        if pd.notna(value):

            parts.append(
                f"{column}: {value}"
            )

    return " | ".join(parts)


texts = df.apply(
    row_to_text,
    axis=1
).tolist()


# ============================================================
# 10. CREATE TRAINING CORPUS
# ============================================================

corpus = "\n<ROW>\n".join(
    texts
)


print("\n====================================")
print("CORPUS INFORMATION")
print("====================================")

print(
    "Corpus characters:",
    len(corpus)
)

print(
    "Number of records:",
    len(texts)
)


print("\nExample training text:\n")

print(
    corpus[:1000]
)


# ============================================================
# 11. CHARACTER TOKENIZER
# ============================================================

characters = sorted(
    list(
        set(corpus)
    )
)

vocab_size = len(
    characters
)


stoi = {
    ch: i
    for i, ch
    in enumerate(characters)
}


itos = {
    i: ch
    for ch, i
    in stoi.items()
}


def encode(text):

    return [
        stoi[ch]
        for ch in text
        if ch in stoi
    ]


def decode(token_ids):

    return "".join(
        itos[int(i)]
        for i in token_ids
    )


# ============================================================
# 12. ENCODE CORPUS
# ============================================================

data = torch.tensor(
    encode(corpus),
    dtype=torch.long
)


print(
    "\nVocabulary size:",
    vocab_size
)

print(
    "Total tokens:",
    len(data)
)


# ============================================================
# 13. TRAIN / VALIDATION SPLIT
# ============================================================

split_index = int(
    0.90 * len(data)
)


train_data = data[
    :split_index
]

val_data = data[
    split_index:
]


print(
    "Training tokens:",
    len(train_data)
)

print(
    "Validation tokens:",
    len(val_data)
)


# ============================================================
# 14. CHECK CORPUS SIZE
# ============================================================

if len(train_data) <= block_size + 1:

    raise ValueError(
        f"Training corpus too small for "
        f"block_size={block_size}. "
        f"Training tokens={len(train_data)}"
    )


if len(val_data) <= block_size + 1:

    print(
        "\nWARNING:"
        " Validation data is smaller than "
        "block_size."
    )

    print(
        "Training data will be used "
        "for validation batches."
    )


# ============================================================
# 15. GET TRAINING BATCH
# ============================================================

def get_batch(split):

    if split == "train":

        source = train_data

    else:

        if len(val_data) > block_size + 1:

            source = val_data

        else:

            source = train_data


    max_start = (
        len(source)
        - block_size
        - 1
    )


    indices = torch.randint(
        0,
        max_start,
        (batch_size,)
    )


    x = torch.stack([
        source[
            i:i + block_size
        ]
        for i in indices
    ])


    y = torch.stack([
        source[
            i + 1:
            i + block_size + 1
        ]
        for i in indices
    ])


    return (
        x.to(device),
        y.to(device)
    )


# ============================================================
# 16. TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(
    nn.Module
):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        dropout
    ):

        super().__init__()


        self.ln1 = nn.LayerNorm(
            embedding_dim
        )

        self.ln2 = nn.LayerNorm(
            embedding_dim
        )


        self.attention = (
            nn.MultiheadAttention(
                embed_dim=
                    embedding_dim,

                num_heads=
                    num_heads,

                dropout=
                    dropout,

                batch_first=True
            )
        )


        self.feed_forward = (
            nn.Sequential(

                nn.Linear(
                    embedding_dim,
                    4 * embedding_dim
                ),

                nn.GELU(),

                nn.Linear(
                    4 * embedding_dim,
                    embedding_dim
                ),

                nn.Dropout(
                    dropout
                )
            )
        )


    def forward(
        self,
        x
    ):

        T = x.size(1)


        causal_mask = (
            torch.triu(

                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                ),

                diagonal=1
            )
        )


        normalized = (
            self.ln1(x)
        )


        attention_output, _ = (
            self.attention(

                normalized,
                normalized,
                normalized,

                attn_mask=
                    causal_mask,

                need_weights=False
            )
        )


        x = (
            x
            + attention_output
        )


        x = (
            x
            + self.feed_forward(
                self.ln2(x)
            )
        )


        return x


# ============================================================
# 17. LANGUAGE MODEL
# ============================================================

class TinyCSVLLM(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers,
        block_size,
        dropout
    ):

        super().__init__()


        self.vocab_size = (
            vocab_size
        )

        self.embedding_dim = (
            embedding_dim
        )

        self.block_size = (
            block_size
        )


        self.token_embedding = (
            nn.Embedding(
                vocab_size,
                embedding_dim
            )
        )


        self.position_embedding = (
            nn.Embedding(
                block_size,
                embedding_dim
            )
        )


        self.blocks = (
            nn.ModuleList([
                TransformerBlock(
                    embedding_dim,
                    num_heads,
                    dropout
                )

                for _ in range(
                    num_layers
                )
            ])
        )


        self.final_norm = (
            nn.LayerNorm(
                embedding_dim
            )
        )


        self.lm_head = (
            nn.Linear(
                embedding_dim,
                vocab_size
            )
        )


    def forward(
        self,
        idx,
        targets=None
    ):

        B, T = idx.shape


        if T > self.block_size:

            raise ValueError(
                f"Sequence length "
                f"{T} exceeds "
                f"block_size "
                f"{self.block_size}"
            )


        positions = (
            torch.arange(
                T,
                device=idx.device
            )
        )


        token_embeddings = (
            self.token_embedding(
                idx
            )
        )


        position_embeddings = (
            self.position_embedding(
                positions
            )
        )


        x = (
            token_embeddings
            + position_embeddings
        )


        for block in self.blocks:

            x = block(x)


        x = (
            self.final_norm(x)
        )


        logits = (
            self.lm_head(x)
        )


        loss = None


        if targets is not None:

            B, T, C = (
                logits.shape
            )


            logits_flat = (
                logits.reshape(
                    B * T,
                    C
                )
            )


            targets_flat = (
                targets.reshape(
                    B * T
                )
            )


            loss = (
                F.cross_entropy(
                    logits_flat,
                    targets_flat
                )
            )


        return logits, loss


    # ========================================================
    # TEXT GENERATION
    # ========================================================

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens=300,
        temperature=0.8,
        top_k=20
    ):

        self.eval()


        for _ in range(
            max_new_tokens
        ):


            idx_context = (
                idx[
                    :,
                    -self.block_size:
                ]
            )


            logits, _ = self(
                idx_context
            )


            logits = (
                logits[
                    :,
                    -1,
                    :
                ]
            )


            logits = (
                logits
                / temperature
            )


            if top_k is not None:

                k = min(
                    top_k,
                    logits.size(-1)
                )


                values, _ = (
                    torch.topk(
                        logits,
                        k
                    )
                )


                cutoff = (
                    values[:, [-1]]
                )


                logits = (
                    torch.where(

                        logits
                        < cutoff,

                        torch.full_like(
                            logits,
                            float("-inf")
                        ),

                        logits
                    )
                )


            probabilities = (
                F.softmax(
                    logits,
                    dim=-1
                )
            )


            next_token = (
                torch.multinomial(
                    probabilities,
                    num_samples=1
                )
            )


            idx = torch.cat(
                [
                    idx,
                    next_token
                ],
                dim=1
            )


        return idx


# ============================================================
# 18. INITIALIZE MODEL
# ============================================================

model = TinyCSVLLM(

    vocab_size=
        vocab_size,

    embedding_dim=
        embedding_dim,

    num_heads=
        num_heads,

    num_layers=
        num_layers,

    block_size=
        block_size,

    dropout=
        dropout

).to(device)


# ============================================================
# 19. COUNT PARAMETERS
# ============================================================

total_parameters = sum(
    p.numel()
    for p
    in model.parameters()
)


trainable_parameters = sum(

    p.numel()

    for p
    in model.parameters()

    if p.requires_grad
)


print("\n====================================")
print("MODEL INFORMATION")
print("====================================")


print(
    f"Total parameters: "
    f"{total_parameters:,}"
)


print(
    f"Model size: "
    f"{total_parameters / 1_000_000:.3f} M"
)


print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)


print(
    "Transformer layers:",
    num_layers
)


print(
    "Embedding dimension:",
    embedding_dim
)


print(
    "Attention heads:",
    num_heads
)


print(
    "FFN dimension:",
    embedding_dim * 4
)


print(
    "Context length:",
    block_size
)


print(
    "Vocabulary size:",
    vocab_size
)


# ============================================================
# 20. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=
        learning_rate,

    weight_decay=
        weight_decay
)


# ============================================================
# 21. EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}


    for split in [
        "train",
        "val"
    ]:


        losses = torch.zeros(
            eval_iters
        )


        for k in range(
            eval_iters
        ):


            xb, yb = (
                get_batch(
                    split
                )
            )


            _, loss = model(
                xb,
                yb
            )


            losses[k] = (
                loss.item()
            )


        results[split] = (
            losses.mean().item()
        )


    model.train()


    return results


# ============================================================
# 22. TRAIN MODEL
# ============================================================

print("\n====================================")
print("TRAINING")
print("====================================")


model.train()


best_val_loss = (
    float("inf")
)


for step in range(
    training_steps + 1
):


    # ========================================================
    # EVALUATION
    # ========================================================

    if (
        step
        % eval_interval
        == 0

        or

        step
        == training_steps
    ):


        losses = (
            estimate_loss()
        )


        train_loss = (
            losses["train"]
        )


        val_loss = (
            losses["val"]
        )


        perplexity = (
            math.exp(
                min(
                    val_loss,
                    20
                )
            )
        )


        print(

            f"Step {step:5d}"

            f" | Train Loss: "
            f"{train_loss:.4f}"

            f" | Val Loss: "
            f"{val_loss:.4f}"

            f" | Perplexity: "
            f"{perplexity:.2f}"
        )


        # Save best weights
        if (
            val_loss
            < best_val_loss
        ):


            best_val_loss = (
                val_loss
            )


            torch.save(
                model.state_dict(),
                WEIGHTS_PATH
            )


    if (
        step
        == training_steps
    ):

        break


    # ========================================================
    # TRAINING STEP
    # ========================================================

    xb, yb = (
        get_batch(
            "train"
        )
    )


    _, loss = model(
        xb,
        yb
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    loss.backward()


    torch.nn.utils.clip_grad_norm_(

        model.parameters(),

        max_norm=1.0
    )


    optimizer.step()


# ============================================================
# 23. SAVE FULL CHECKPOINT
# ============================================================

checkpoint = {

    "model_state_dict":
        model.state_dict(),

    "optimizer_state_dict":
        optimizer.state_dict(),

    "stoi":
        stoi,

    "itos":
        itos,

    "vocab_size":
        vocab_size,

    "embedding_dim":
        embedding_dim,

    "num_heads":
        num_heads,

    "num_layers":
        num_layers,

    "block_size":
        block_size,

    "dropout":
        dropout,

    "total_parameters":
        total_parameters,

    "training_steps":
        training_steps,

    "learning_rate":
        learning_rate,

    "best_val_loss":
        best_val_loss,

    "csv_columns":
        list(df.columns),

    "csv_files":
        [
            file.name
            for file in csv_files
        ]
}


torch.save(
    checkpoint,
    CHECKPOINT_PATH
)


# ============================================================
# 24. SAVE MODEL WEIGHTS
# ============================================================

torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


print("\n====================================")
print("MODEL SAVED")
print("====================================")


print(
    "Checkpoint:",
    CHECKPOINT_PATH
)


print(
    "Weights:",
    WEIGHTS_PATH
)


# ============================================================
# 25. GENERATE TEXT
# ============================================================

model.eval()


prompt = "model_type:"


prompt_ids = encode(
    prompt
)


if len(prompt_ids) == 0:

    raise ValueError(
        "Prompt contains no known "
        "characters."
    )


input_tensor = (
    torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device
    )
)


generated_tokens = (
    model.generate(

        input_tensor,

        max_new_tokens=300,

        temperature=0.8,

        top_k=20
    )
)


generated_text = decode(
    generated_tokens[
        0
    ].tolist()
)


print("\n====================================")
print("GENERATED TEXT")
print("====================================")


print(
    generated_text
)


# ============================================================
# 26. RELOAD SAVED MODEL
# ============================================================

print("\n====================================")
print("RELOADING MODEL")
print("====================================")


loaded_checkpoint = torch.load(

    CHECKPOINT_PATH,

    map_location=device
)


loaded_stoi = (
    loaded_checkpoint[
        "stoi"
    ]
)


loaded_itos = (
    loaded_checkpoint[
        "itos"
    ]
)


loaded_model = TinyCSVLLM(

    vocab_size=
        loaded_checkpoint[
            "vocab_size"
        ],

    embedding_dim=
        loaded_checkpoint[
            "embedding_dim"
        ],

    num_heads=
        loaded_checkpoint[
            "num_heads"
        ],

    num_layers=
        loaded_checkpoint[
            "num_layers"
        ],

    block_size=
        loaded_checkpoint[
            "block_size"
        ],

    dropout=
        loaded_checkpoint[
            "dropout"
        ]

).to(device)


loaded_model.load_state_dict(

    loaded_checkpoint[
        "model_state_dict"
    ]
)


loaded_model.eval()


print(
    "Loaded parameters:",
    f"{loaded_checkpoint['total_parameters']:,}"
)


print(
    "Model successfully loaded."
)


# ============================================================
# 27. TOKENIZER FOR LOADED MODEL
# ============================================================

def loaded_encode(text):

    return [
        loaded_stoi[ch]
        for ch in text
        if ch in loaded_stoi
    ]


def loaded_decode(ids):

    return "".join(
        loaded_itos[int(i)]
        for i in ids
    )


# ============================================================
# 28. GENERATION USING RELOADED MODEL
# ============================================================

prompt = "model_type:"


prompt_ids = loaded_encode(
    prompt
)


x = torch.tensor(

    [prompt_ids],

    dtype=torch.long,

    device=device
)


generated = (
    loaded_model.generate(

        x,

        max_new_tokens=300,

        temperature=0.8,

        top_k=20
    )
)


result = loaded_decode(
    generated[
        0
    ].tolist()
)


print("\n====================================")
print("OUTPUT FROM RELOADED MODEL")
print("====================================")


print(
    result
)

Device: cuda

CSV FILES FOUND
33673b86.csv
9f56743a.csv
merged.csv

Total CSV files: 3

COLUMN CHECK
Reference file: 33673b86.csv
Reference columns: 74

33673b86.csv: 74 columns
  Columns match

9f56743a.csv: 74 columns
  Columns match

merged.csv: 74 columns
  Columns match

All CSV files have compatible columns.

READING CSV FILES
33673b86.csv: 50 rows, 74 columns
9f56743a.csv: 4 rows, 74 columns
merged.csv: 12 rows, 74 columns

MERGED DATASET
Total CSV files: 3
Total rows: 66
Total columns: 74

Column names:
 - timestamp
 - unique_device_id
 - device_short_id
 - pc_name
 - collection_mode
 - sample_index
 - true_label
 - prediction
 - correct
 - model_type
 - parameters
 - model_flops
 - confidence_score
 - logit_margin
 - entropy
 - execution_time_sec
 - cpu_energy_kwh
 - gpu_energy_kwh
 - ram_energy_kwh
 - total_energy_kwh
 - total_emissions_kg
 - carbon_intensity_kgco2_kwh
 - codecarbon_version
 - input_tokens
 - output_tokens
 - total_tokens
 - tokens_per_second
 - joules_per_to

# Load the model

In [2]:
import torch

CHECKPOINT_PATH = "csv_llm_1M_checkpoint.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

# Load tokenizer
stoi = checkpoint["stoi"]
itos = checkpoint["itos"]

# Rebuild model
model = TinyCSVLLM(
    vocab_size=checkpoint["vocab_size"],
    embedding_dim=checkpoint["embedding_dim"],
    num_heads=checkpoint["num_heads"],
    num_layers=checkpoint["num_layers"],
    block_size=checkpoint["block_size"],
    dropout=checkpoint["dropout"]
).to(device)

# Load trained weights
model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Model loaded successfully.")
print("Parameters:", checkpoint["total_parameters"])

Model loaded successfully.
Parameters: 1026761


In [5]:
# ============================================================
# RUN PROMPT
# ============================================================

def encode(text):
    return [
        stoi[ch]
        for ch in text
        if ch in stoi
    ]

def decode(ids):
    return "".join(
        itos[int(i)]
        for i in ids
    )


prompt = """
model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
task: predict approximate execution time |
execution_time_sec:
"""


# Convert prompt to token IDs
prompt_ids = encode(prompt)

x = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)


# Generate output
with torch.no_grad():
    generated = model.generate(
        x,
        max_new_tokens=100,
        temperature=0.6,
        top_k=10
    )


# Decode generated tokens
result = decode(
    generated[0].tolist()
)

print("\n==============================")
print("MODEL OUTPUT")
print("==============================")

print(result)


MODEL OUTPUT

model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
task: predict approximate execution time |
execution_time_sec:
re: 1.00006 | cpu_energy_kwh: 1.293331838683119e-07 | gpu_energy_kwh: 1.77683772206e-08 | ram_energy


In [8]:
prompt = """
model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
execution_time_sec:
"""

result = decode(
    generated[0].tolist()
)

# Get only text after execution_time_sec:
prediction = result.split("execution_time_sec:")[-1].strip()

# Take only the first generated value
prediction = prediction.split("|")[0].strip()
prediction = prediction.split("\n")[0].strip()

print('Execution time (sec): {} s'.format(prediction))

Execution time (sec): re: 1.00006 s
